# 12. Views and Copies

This is probably the most common source of confusing bugs in NumPy code. Some operations give you a **view** that shares memory with the original array, others give you an independent **copy**. Modify a view and the original changes too.

In [1]:
import numpy as np

## No copy at all: assignment

`b = a` doesn't create a new array. Both names point to the same object.

In [2]:
a = np.arange(5)
b = a
b[0] = 100
print(a)
print(b is a)

[100   1   2   3   4]
True


## Views

A view is a new array object that looks at the same data. Basic slicing returns a view:

In [3]:
a = np.arange(10)
s = a[2:5]
s[:] = -1
print(a)

[ 0  1 -1 -1 -1  5  6  7  8  9]


You can check with `np.shares_memory`, or look at `.base` (a view's `base` is the array it came from; a fresh array has `base = None`):

In [4]:
print(np.shares_memory(a, s))
print(s.base is a)
print(a.base)

True
True
None


## Copies

Explicit `.copy()`, plus boolean and fancy indexing, always give you a copy:

In [5]:
a = np.arange(10)

c = a[2:5].copy()
m = a[a > 5]
f = a[[1, 3, 5]]

c[:] = 0; m[:] = 0; f[:] = 0
print(a)     # untouched

[0 1 2 3 4 5 6 7 8 9]


## Cheat sheet

| Operation | Result |
|---|---|
| `b = a` | same object |
| `a[1:5]`, `a[::2]`, `a[:, 0]` | view |
| `a.T`, `a.transpose()`, `a.swapaxes()` | view |
| `a.reshape(...)` | view if possible, otherwise copy |
| `a.ravel()` | view if possible, otherwise copy |
| `a.view(dtype)` | view |
| `np.newaxis`, `squeeze`, `expand_dims` | view |
| `a.flatten()` | copy |
| `a.copy()`, `np.copy(a)`, `np.array(a)` | copy |
| `a[[0, 2]]` (fancy indexing) | copy |
| `a[a > 0]` (boolean indexing) | copy |
| `a + 1`, `np.sqrt(a)`, ... | new array |
| `a.astype(...)` | copy (by default) |

## Where this bites

### 1. Modifying a slice you thought was separate

In [6]:
data = np.array([3., 1., 4., 1., 5.])
first_three = data[:3]
first_three /= first_three.max()     # meant to normalise just this piece
print(data)                          # ...but the original changed too

[0.75 0.25 1.   1.   5.  ]


Fix: call `.copy()` when you plan to modify something and want the original kept intact.

### 2. Assignment with chained indexing

Assigning through a fancy index followed by a slice doesn't work, because the first step already made a copy:

In [7]:
a = np.zeros((3, 3))
a[[0, 2]][:, 0] = 1    # modifies a temporary copy, not a
print(a)

a[[0, 2], 0] = 1       # do it in one step instead
print(a)

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
[[1. 0. 0.]
 [0. 0. 0.]
 [1. 0. 0.]]


Boolean masks work fine in a single assignment, since NumPy turns `a[mask] = value` into a direct write:

In [8]:
a = np.arange(6)
a[a % 2 == 0] = -1
a

array([-1,  1, -1,  3, -1,  5])

### 3. Keeping a huge array alive

A small view keeps the **whole** original array in memory, because it still points into its buffer. If you only need a small piece of a big array, copy it so the big one can be freed.

In [9]:
big = np.ones(10_000_000)
small_view = big[:10]
small_copy = big[:10].copy()
print(small_view.base is big, small_copy.base is None)

True True


## When views are what you want

Views are fast and memory-efficient, and sometimes they're exactly what you need. For example, a sliding window over a time series without copying anything:

In [10]:
from numpy.lib.stride_tricks import sliding_window_view

x = np.arange(10)
windows = sliding_window_view(x, 3)
print(windows)
print('moving average:', windows.mean(axis=1))
print(np.shares_memory(x, windows))

[[0 1 2]
 [1 2 3]
 [2 3 4]
 [3 4 5]
 [4 5 6]
 [5 6 7]
 [6 7 8]
 [7 8 9]]
moving average: [1. 2. 3. 4. 5. 6. 7. 8.]
True


`.view(dtype)` reinterprets the same bytes as a different type. It's low level and you won't need it much, but it shows that a view really is the same memory:

In [11]:
f = np.array([1.0], dtype=np.float64)
print(f.view(np.int64))     # the raw bit pattern of 1.0

[4607182418800017408]
